# Homework 8 Solutions - Partial Differential Equation Integrators (10 pt)

Many systems in astrophysics --- planets, stars, accretion disks, nebulae, galaxies, the universe --- can be modelled with sets of fluid equations. These equations cannot typically be solved analytically, especially with all the effects one needs to take into account if one wants an accurate representation of the physics involved, but must be solved numerically. This homework assignment uses two key PDE solution methods that we learned in class to solve to important PDEs: the Sedov Exoplosion and the diffusion equation for a viscous accretion disk.

**Instructions:** Read the descriptor for each problem, then write your solution code in the Jupyter code cell. Add your homework 8 assignment to the list of things to modify with `git add hw8.ipynb`, and commit changes in your assignment as frequently as you save your homework assignment. Once you are done, commit your finishing changes with the command `git commit -m "Completed hw 8"`, then push/submit to the instructor/TA using `git push`. If you push your assignment more than once, this is fine! Your most recently pushed assignment will be the one that is graded.

To accept this assignment, click the link below and follow the steps:\
https://classroom50.org/PsuAstro410/410astro26/assignments/homework-8/accept

To also download the python script, which you will import to use pre-imported modular features, as well as write your own additions to the script, run the following command below:
```bash
curl -O https://raw.githubusercontent.com/PsuAstro410/psuastro410.github.io/refs/heads/main/homeworks/hw8_module.py
```

In [ ]:
# Nice things for this homework assignment
import matplotlib.pyplot as plt
import numpy as np
import hw8_module as hwmod
from scipy import linalg

# Problem 1 - Sedov Explosion and Spherical Symmetry (5 pt)

We solved the equations of hydrodynamics in 1-d Cartesian coordinates.
If we want to model something that is spherically symmetric, we can do
1-d spherical coordinates.

In 1-d spherical coordinates, the equations of hydrodynamics are:

\begin{align*}
\frac{\partial \rho}{\partial t} + \frac{1}{r^2} \frac{\partial (r^2 \rho u)}{\partial r} &= 0 \\
\frac{\partial (\rho u)}{\partial t} + \frac{1}{r^2} \frac{\partial (r^2 \rho u^2)}{\partial r} + \frac{\partial p}{\partial r} &= 0 \\
\frac{\partial (\rho E)}{\partial t} + \frac{1}{r^2} \frac{\partial }{\partial r} \left [ r^2 (\rho E + p) u \right ] &= 0
\end{align*}

The main difference is that the divergence has area and volume terms now.

A good problem to try with this is the Sedov blast wave explosion.  In this problem, you put a lot of energy into a point at the center (the
origin of our spherical coordinates) and watch a spherical blast wave move outward.

As it turns out, the Sedov blast wave solution does a great job approximating the behavior of a supernovae explosion. It also does a good job approximating the behavior after the sudden detonation of an atomic bomb. I advise not discussing this interesting fact when going through TSA if you are travelling home for Thanksgiving...

There is an analytic solution for a gamma-law gas that can be compared to. We implement this solution, and plot it in the cells below. You will later compare this solution to the result from your numerical calculations:

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(6, 10), constrained_layout=True)
gamma = 5/3

exact = hwmod.SedovExact(gamma)
xfine = np.linspace(1e-3, 1.3, 3000)
colors = ['C0', 'C1', 'C2']

for n, tval in enumerate([0.25, 0.5, 1.0]):
   for i, profile in enumerate(exact.profiles(xfine, tval)):
        ax[i].plot(xfine, profile, color=colors[n], lw=1.0, ls='--', label=f"exact, t={tval:.2f}")

for i in range(3):
  ax[i].legend(fontsize=8)
ax[2].set_xlabel('$r$')
ax[0].set_ylabel(r'$\rho$')
ax[1].set_ylabel('$u$')
ax[2].set_ylabel('$p$')

plt.show()

## Exercise 1(a) - Modify the `SphericalFluid1DGrid` class to include reflecting inner and outflow outer boundary conditions (1 pt)

In `hw8_module.py`, you might notice that the `SphericalFluid1DGrid` class is similar to `Fluid1DGrid` class from lecture 20, except we now also have attributes added to calculate the area and volume of grid cells. Importantly, we also have to change the boundary conditions to be *reflecting* boundary conditions. Intuitively, we want these conditions to capture if material travels towards the center, it will all pile-up as $r \to 0$, so this material should "bounce" when it hits the stellar center.

Specifically, the ghost cells at the inner boundary has the same value as the corresponding zone in the interior (e.g., same distance away from the boundary).  But for the velocity, we switch the sign, such that the velocity moving toward the interface will go to zero.

This would look like:

   $$\begin{align*}
      \rho_{\mathrm{lo}-1} &= \rho_{\mathrm{lo}} \\
      \rho_{\mathrm{lo}-2} &= \rho_{\mathrm{lo+1}} \\
   \end{align*}$$

   $$\begin{align*}
      (\rho u)_{\mathrm{lo}-1} &= -(\rho u)_{\mathrm{lo}} \\
      (\rho u)_{\mathrm{lo}-2} &= -(\rho u)_{\mathrm{lo+1}} \\
   \end{align*}$$

   $$\begin{align*}
      (\rho E)_{\mathrm{lo}-1} &= (\rho E)_{\mathrm{lo}} \\
      (\rho E)_{\mathrm{lo}-2} &= (\rho E)_{\mathrm{lo+1}} \\
   \end{align*}$$

Notice if we re-wrote this for the primitive variables, their form would be very similar:

   $$\begin{align*}
      \rho_{\mathrm{lo}-1} &= \rho_{\mathrm{lo}} \\
      \rho_{\mathrm{lo}-2} &= \rho_{\mathrm{lo+1}} \\
   \end{align*}$$

   $$\begin{align*}
      (u)_{\mathrm{lo}-1} &= -(u)_{\mathrm{lo}} \\
      (u)_{\mathrm{lo}-2} &= -(u)_{\mathrm{lo+1}} \\
   \end{align*}$$

   $$\begin{align*}
      (p)_{\mathrm{lo}-1} &= (p)_{\mathrm{lo}} \\
      (p)_{\mathrm{lo}-2} &= (p)_{\mathrm{lo+1}} \\
   \end{align*}$$

Therefore, as long as we write this boundary condition, loading a general type of variables into the grid structure, we should be fine.

For the outer grid, we will just assume an outflow boundary condition, which for all variables $a$ and integers `i<ng`, looks like
$$
(a)_{\mathrm{hi}+i} = a_{\mathrm{hi}}
$$

In the `SphericalFluid1DGrid` class, modify the `fill_BCs(self,a):` attribute to implement reflecting boundary conditions at the inner boundary, and outflow boundary conditions at the outer boundary. Call all the inner ghost cells with a command like
```python
a[i, :self.ilo] = 
```
with `i=0,1,2` depending on the variable you are assigning, and set these variables equal to the values of `a` on the active grid with a command like
```python
<variable value>[i, self.ilo:self.ilo+self.ng][::-1]
```
where we are looping up to `ng` values of the grid, and flipping the ordering with `::-1`.



Edit the code in the `hw8_module.py` file, and paste your edits into the markdown cell below:

**Answer here**

```python

    # FILL IN CODE BELOW FOR EXERCISE 1(a)

```

## Exercise 1(b) - Add geometrical terms to the timestep update (1 pt)

The Euler Equations in spherical coordinates can be re-arranged into a conservative form, leading to the relations
\begin{align*}
\frac{\partial \rho}{\partial t} + \frac{1}{r^2} \frac{\partial (r^2 \rho u)}{\partial r} &= 0 \\
\frac{\partial (\rho u)}{\partial t} + \frac{1}{r^2} \frac{\partial (r^2 \rho u^2 + r^2 p)}{\partial r} &= \frac{2}{r} p \\
\frac{\partial (\rho E)}{\partial t} + \frac{1}{r^2} \frac{\partial }{\partial r} \left [ r^2 (\rho E + p) u \right ] &= 0
\end{align*}
Defining our conservative variables
$$
{\bf U} = \left ( \begin{array}{c} \rho \\ \rho u \\ \rho E \end{array} \right )
$$
and the flux ${\bf F}({\bf U})$
$$
{\bf F} = \left ( \begin{array}{c} \rho u \\ \rho u^2 + p  \\ (\rho E + p) u \end{array} \right )
$$
as before, we obtain
$$
\frac{\partial {\bf U}}{\partial t} + \frac{1}{r^2}\frac{\partial (r^2 {\bf F})}{\partial r} = {\bf S},
$$
except now the partial of the flux also has an $r^2$ term due to our coordinate system having more area as $r$ increases, with an additional source term
$$
{\bf S} = \left ( \begin{array}{c} 0 \\ \frac{2}{r} p \\ 0 \end{array} \right ).
$$

To figure out our finite-value form for these equations at cell $i$, we calculate the average fluxes moving in and out of the domain by integrate both sides by $4\pi r^2 dr$ from interface $i -1/2$ to $i + 1/2$, and divide by the volume $V_i$ to give
$$
\frac{\partial {\bf U}_i}{\partial t} + \frac{A_{i + 1/2} {\bf F}_{i+1/2} - A_{i - 1/2} {\bf F}_{i-1/2}}{V_i} = {\bf S}_i,
$$
where
$$
{\bf S}_i = \left ( \begin{array}{c} 0 \\ p_i (A_{i+1/2} - A_{i-1/2})/V_i \\ 0 \end{array} \right ).
$$

Taking the forward timestep
$$
\frac{\partial {\bf U}_i}{\partial t} \approx \frac{{\bf U}_i^{n+1} - {\bf U}_i^n}{\Delta t},
$$
we have the explicit update for our new timestep
$$
{\bf U}_i^{n+1} = {\bf U}_i^{n} - \frac{\Delta t}{V_i}(A_{i + 1/2} {\bf F}_{i+1/2} - A_{i - 1/2} {\bf F}_{i-1/2}) + \Delta t \cdot {\bf S}_i.
$$

In the portion indicated in `hw8_module.py` within `euler_fv_spherical`, modify the Flux at interface states ${\bf F}_{i \pm 1/2}$ and variable ${\bf U}^{n+1}_i$ according to the code written above. In addition to modifying the module, copy and past the edits in the markdown cell below. 

**Note:** The grid class `SphericalFluid1DGrid` calculates the area $A_{i+1/2}$ of the interface at $r_{i+1/2}$, and volume $V_i$ of zone $i$, with the lines
```python
        self.A = 4*np.pi*self.xp**2
        self.V = 4*np.pi/3*(self.xp**3 - self.xm**3)
```
If you use these attributes in your solution (since the grid has already been made an object in the code), your solution will be much more tidy.

**Answer here**

```python
\        # F[:, i] is the flux at x_{i+1/2}
        for i in range(g.ilo-g.ng, g.ihi+g.ng):
            # FILL IN CODE BELOW FOR EXERCISE 1(b)
            F[:, i] = 

        for i in range(g.ilo, g.ihi+1):
            # flux difference, weighted by the interface areas
            # FILL IN CODE BELOW FOR EXERCISE 1(b)
            Unew[:, i] = 

```

## Exercise 1(c) - Sedov Initial Conditions (1 pt)

Setup the Sedov problem, run, and compare to the exact solution. For the initial conditions, it is common to specify an initial radius of the explosion, $r_0$.  Then initialize the pressure as:
$$
p = \left \{ \begin{array}{cc} p_\mathrm{expl} & r \le r_0 \\
            p_\mathrm{ambient} & r > r_0 \end{array} \right .
$$
with
$$
p_\mathrm{ambient} = 10^{-5}
$$
and
$$
p_\mathrm{expl} = (\gamma -1) \frac{E_\mathrm{Sedov}}{(4/3) \pi r_0^3}
$$
and $E_\mathrm{Sedov} = 1$ is the initial explosion energy.  This formulation finds the pressure corresponding to spreading that energy over a sphere of radius $r_0$.

The initial velocity is set to 0.0:
$$
u = 0
$$

The density everywhere is 1.0:
$$
\rho = 1
$$


In [ ]:
## ANSWER BELOW



## Exercise 1(d) - Plot Sedov solution (2 pt)

Evolve the sedov solution with $\gamma = 5/3$, $200$ radial gridpoints, $C = 0.3$, up to times $t = 0.25, 0.5, 1.0$ in these units. Plot the numerical solution $\rho$, $u$, and $p$ over the spatial grid.

Compare these solutions to the exact profiles calculated analytically. To do so, use the `SedovExact` class in the `hw8_module` module. This module returns profiles for $\rho$, $u$, and $p$ over a spatial grid $x$ at a time `tval` with the `profiles` attribute.

In [ ]:
## ANSWER BELOW

# Viscous Accretion Disk time evolution (5 pt)

When cold gas orbits a massive body like a planet, star, or black hole, the vertical direction is much less extended than the horizontal one. This is because the orbital motion around the central body can keep material supported in the horizontal direction, while gas pressure can only weakly support material in the vertical direction. To understand the evolution of this *accretion disk*, one assumes azimuthal symmetry, and vertically averages over the density
$$
\Sigma(R, t) = \int_{-\infty}^{\infty} \rho (R, z, t) dz,
$$
and calculates the evolution of the surface density $\Sigma(r, t)$. When the disk is subject to a "viscosity" $\nu$, the time evolution can be described by an equation of the form (e.g. Pringle 1981)
$$
\frac{\partial \Sigma}{\partial t} = \frac{3}{R} \frac{\partial}{\partial R} \left[ R^{1/2} \frac{\partial}{\partial R} (\nu R^{1/2} \Sigma) \right]. \qquad (1)
$$
For a review, see the reference below: \
https://arxiv.org/html/2201.07262v1#S3.E153 

We could solve this equation directly with a PDE solver, but it would just take more algebra. Instead, we will use a few math tricks to get this equation in a form that is a bit simpler using a change of variables: define
$$
\phi = R^{1/2} \Sigma, \qquad x = R^{1/2},
$$
and the equation becomes
$$
\frac{\partial \phi}{\partial t} = k \frac{\partial^2 \phi}{\partial x^2}, \qquad (2)
$$
where
$$
k(x) = \frac{3 \nu}{4 x^2},
$$
and we have assumed the viscosity $\nu = \mathrm{constant}$ for simplicity. In this exercise, we will solve this equation using the Crank-Nicolson method.

## Exercise 2(a) - Create the Disk1DGrid class (2 pt)

In analogy with our diffusion equation grid class we discussed in class, create a `Disk1DGrid` class, with inputs `nx, ng, Rmin, Rmax` (with `Rmin, Rmax` the physical radii of the disk). Not only define attributes `ng, nx, Rmin, Rmax`, but also `xmin, xmax` corresponding to the variable we actually use to solve the system $x$, as well as `x` and `dx = (xmax-xmin)/(nx)`. Store the variable `phi` that we use to calculate the surface density evolution as a class attribute. Calculate surface density itself `Sigma = phi/R**0.5` with a method inside of the class. Also define a method that returns a scratch array, as we have defined for our other classes.

For `fill_BCs`, create a method that calculates a zero-torque inner boundary condition, and a no-infall outer boundary condition. In terms of physical variables, this translates to forcing
$$
(\phi)_{{\rm lo} - i} = -(\phi)_{{\rm lo} + i - 1}
$$
at the inner boundary, and 
$$
(\phi)_{{\rm hi} + i} = (\phi)_{{\rm hi}}
$$
at the outer boundary, for $i = 1, \ldots, n_g$.

In addition to modifying the `hw8_modules.py` module, copy and paste your edits to the code below.

**Answer here**

```python
class Disk1DGrid:

    """a finite-volume grid for disk variables in cylindrical symmetry"""
    # FILL IN CODE BELOW FOR EXERCISE 2(a)
```

## Exercise 2(b) - Writing the implicit step function (2 pt)

Using the discritization scheme that we discussed in the class exercises, equation (2) can be discritized using the Crank-Nicolson scheme, giving an equation for the updates of
$$
\frac{u_i^{n+1} - u_i^n}{\Delta t} = \frac{k_i}{2} \left( \frac{u_{i+1}^{n+1} - 2 u_i^{n+1} + u_{i-1}^{n+1}}{\Delta x^2} + \frac{u_{i+1}^{n} - 2 u_i^{n} + u_{i-1}^{n}}{\Delta x^2} \right),
$$
where
$$
k_i = k(x_i).
$$
Defining
$$
\alpha_i = \frac{k_i \Delta t}{\Delta x^2},
$$
we can re-write this equation as
$$
-\frac{\alpha_i}{2} u_{i-1}^{n+1} + (1 + \alpha_i) u_i^{n+1} - \frac{\alpha_i}{2} u^{n+1}_{i+1} = \frac{\alpha_i}{2} u_{i-1}^{n} + (1 - \alpha_i) u_i^{n} + \frac{\alpha_i}{2} u^{n}_{i+1}.
$$

The cells near the ghost cells are determined by the boundary conditions. The inner zero-torque boundary condition implies that we want $\phi = 0$ at the inner cells, implying that the ghost cells must satisfy
$$
\phi_{{\rm lo}-1} = -\phi_{\rm lo}.
$$
Inserting this into the difference equation above gives
$$
\left( 1 + \frac{3\alpha_{\rm lo}}{2} \right) \phi^{n+1}_{\rm lo} - \frac{\alpha_{\rm lo}}{2} \phi^{n+1}_{{\rm lo}+1} = \left( 1 + \frac{3\alpha_{\rm lo}}{2} \right) \phi^{n}_{\rm lo} - \frac{\alpha_{\rm lo}}{2} \phi^{n}_{{\rm lo}+1}
$$
The zero-infall boundary condition implies that $\partial \phi/\partial x = 0$ at the outer boundary, implying that
$$
\frac{\phi_{{\rm hi}+1} - \phi_{{\rm hi}}}{\Delta x} = 0 \qquad \Rightarrow \phi_{{\rm hi}+1} = \phi_{{\rm hi}}.
$$
Inserting this into the above difference equation gives
$$
- \frac{\alpha_{\rm hi}}{2} \phi_{{\rm hi}-1}^{n+1} + \left( 1 + \frac{\alpha_{\rm hi}}{2} \right) \phi^{n+1}_{\rm hi} = - \frac{\alpha_{\rm hi}}{2} \phi_{{\rm hi}-1}^{n} + \left( 1 + \frac{\alpha_{\rm hi}}{2} \right) \phi^{n}_{\rm hi}
$$

The right-hand side uses the ghost cells, so fill the boundary conditions before computing it. The ghost cells also change the first and last rows of the matrix. At the inner boundary, $u_{\rm lo-1} = -u_{\rm lo}$, so the diagonal becomes $1 + 3\alpha_{\rm lo}/2$. At the outer boundary, $u_{\rm hi+1} = u_{\rm hi}$, so the diagonal becomes $1 + \alpha_{\rm hi}/2$.

**Note:** because $\alpha_i$ changes from zone to zone, take care which $\alpha$ goes where in the banded matrix. In `solve_banded` storage, `u[i]` multiplies $u_i$ in row $i-1$, and `l[i]` multiplies $u_i$ in row $i+1$, so `u[1:] = -alpha[:-1]/2` and `l[:-1] = -alpha[1:]/2`.

Re-write the `crank_nicolson_step` algorithm you wrote for the in-class exercise, to solve for the next step for the accretion disk algorithm. Write this algorithm in the code cell below, **and do not write in `hw8_module.py`**.

In [ ]:
## ANSWER BELOW


## Exercise 2(c) - Write the initial conditions

Similar to the original papers that examined the evolution of an accretion disk, we will input a disk that is a gaussian bump, and watch this bump viscously spead. Specifically, we will choose as an initial condition for the surface density of
$$
\Sigma(t=0, R) = \exp \left[ -\frac{(R-R_0)^2}{2 \sigma_R^2} \right]
$$
with $R_0 = 10$ and $\sigma_R = 1$. 

Write a function that sets this initial condition, taking a grid object `gr` as an input, and outputs $\Sigma(t=0, R)$.

In [ ]:
## ANSWER BELOW


## Exercise 2(d) - Write integration algorithm (1 pt)

Similar to the in-class exercise, write a function that carries out a Crank-Nicolson integration of this system, up to a time `tmax`. In addition to `tmax`, take as steps the number of radial gridpoints `nx`, viscosity `nu`, Courant number `C`, and initial condition `init_cond`. Pass `Rmin, Rmax` also, with default values `Rmin=1` and `Rmax=100` (we will not modify these for the final step).

In [ ]:
## ANSWER BELOW


## Exercise 2(e) - Plot the solution (1 pt)

In units of the viscous time at the initial disk radius $t_{\rm visc} = R_0^2/\nu$, integrate the viscous disk equation to times $t_{\rm max} = 0.00, 0.01, 0.04, 0.16 \ t_{\rm visc}$, with `nx = 200` grid points, and viscosity $\nu = 1$. Plot the surface density $\Sigma$ versus radius $R$, from $R = R_{\rm min} = 1$ to $R = R_{\rm max} = 100$, with the x-axis scale logarithmic and y-axis linear. 

In [ ]:
## ANSWER BELOW
